#### Scrape Metadata from Web of Science (WOS)

This needs first and last names as input.
Website is opened an LMU Media Login used.
Then a loop starts and we extract bibliography info for the people.

In [11]:
#set up

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options
import time
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import os
import pandas as pd

path_data = "D:/Projektfolder1 (Miethe, Grosenick)/zz_AmelieMisc/thankyou/data/gen"


In [12]:

# -------------------------------------------------------------
# CONFIG
# -------------------------------------------------------------

START_URL = "https://www-webofscience-com.emedien.ub.uni-muenchen.de/wos/woscc/smart-search"

# If you need login, set your credentials here
LMU_USERNAME = "ra56cin"
LMU_PASSWORD = "WiWi1616rote"


In [13]:
# -------------------------------------------------------------
# LOAD LIST OF NAMES TO SCRAPE
# -------------------------------------------------------------

# load
output_file = os.path.join(path_data, "dt_acknowledged_gendered.xlsx")
df_toscrape = pd.read_excel(output_file)

# keep just first and last name (all unique combos of them)
df_toscrape = df_toscrape[['first_name', 'last_name']].drop_duplicates().reset_index(drop=True)

# to test
#df_toscrape = df_toscrape.iloc[[1]]  # double brackets keep it as a DataFrame
#df_toscrape = df_toscrape.head(10)

print(df_toscrape.head(20))


   first_name      last_name
0   ALEXANDRE            MAS
1       DARON       ACEMOGLU
2       DAVID          AUTOR
3       DAVID         DEMING
4       EMILY          OSTER
5      ESTHER          DUFLO
6   FRANCESCO         TREBBI
7      GAUTAM            RAO
8      GEORGY         EGOROV
9       JESSE        SHAPIRO
10       JOHN       FRIEDMAN
11   LAWRENCE           KATZ
12   MARIANNE       BERTRAND
13      MARIO          MACIS
14       NAVA         ASHRAF
15      PAOLA       GIULIANO
16    PATRICK       FRANCOIS
17    RICARDO  PEREZ-TRUGLIA
18   STEPHANE       BONHOMME
19  CHRISTIAN        HELLWIG


In [14]:
# -------------------------------------------------------------
# SETUP SELENIUM
# -------------------------------------------------------------

options = Options()
options.add_argument("--start-maximized")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)


In [15]:
# -------------------------------------------------------------
# STEP 1: Load the Web of Science Smart Search URL
# -------------------------------------------------------------

driver.get(START_URL)
time.sleep(3)


In [16]:
# -------------------------------------------------------------
# STEP 2: Handle LMU EZproxy login (if shown)
# -------------------------------------------------------------
try:
    # Username field
    username_field = driver.find_element(By.ID, "username")
    username_field.send_keys(LMU_USERNAME)

    # Password field
    password_field = driver.find_element(By.ID, "password")
    password_field.send_keys(LMU_PASSWORD)
    password_field.send_keys(Keys.RETURN)

    print("Logged in via LMU ezProxy …")
    time.sleep(5)

except Exception:
    print("No EZproxy login page detected. Continuing …")


Logged in via LMU ezProxy …


In [17]:
# -------------------------------------------------------------
# STEP 3: Reject cookies if the banner appears
# -------------------------------------------------------------

try:
    # Wait a moment for cookie banner to load
    time.sleep(5)
    
    reject_cookies_button = driver.find_element(By.ID, "onetrust-reject-all-handler")
    reject_cookies_button.click()
    
    print("Cookies rejected ✅")
    time.sleep(1)  # allow banner to disappear

except Exception:
    print("No cookie banner detected, continuing …")


Cookies rejected ✅


In [18]:
# -------------------------------------------------------------
# STEP 4: Click on "Advanced Search" tab
# -------------------------------------------------------------

try:
    # Wait until the Advanced Search tab link is clickable
    advanced_tab_link = WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.ID, "snAdvancedLink"))
    )
    advanced_tab_link.click()
    print("Clicked on Advanced Search tab ✅")
    time.sleep(2)  # wait for the Advanced Search interface to load

except Exception as e:
    print("Could not find Advanced Search tab:", e)

Clicked on Advanced Search tab ✅


In [19]:
# -------------------------------------------------------------
# STEP 5: Click on "RESEARCHERS" tab
# -------------------------------------------------------------

try:
    # Wait until the Researchers tab link is clickable
    researchers_tab = WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.ID, "snResearcherSearch"))
    )
    researchers_tab.click()
    print("Clicked on RESEARCHERS tab ✅")
    time.sleep(2)  # wait for the Researchers search interface to load

except Exception as e:
    print("Could not find RESEARCHERS tab:", e)

Clicked on RESEARCHERS tab ✅


In [20]:
import time

# List to store all authors' data
all_authors_data = []

# Track start time
start_time = time.time()

# -------------------------------------------------------------
# Loop over all names in df_toscrape
# -------------------------------------------------------------
for idx, row in df_toscrape.iterrows():
    first_name = row['first_name']
    last_name = row['last_name']
    author_data = {
        'first_name': first_name,
        'last_name': last_name
    }

    # -------------------------------
    # Estimate time remaining
    # -------------------------------
    elapsed = time.time() - start_time
    if idx > 0:
        avg_time_per_author = elapsed / idx
        remaining = avg_time_per_author * (len(df_toscrape) - idx)
        print(f"\nProcessing {first_name} {last_name} ({idx+1}/{len(df_toscrape)})")
        print(f"Elapsed: {elapsed:.1f}s | Estimated remaining: {remaining:.1f}s (~{remaining/60:.1f} min)")
    else:
        print(f"\nProcessing {first_name} {last_name} (1/{len(df_toscrape)})")
        print("Estimating time remaining...")

    # -------------------------------
    # Ensure we are on the search page
    # -------------------------------
    search_page_url = "https://www-webofscience-com.emedien.ub.uni-muenchen.de/wos/author/author-search"
    if not driver.current_url.startswith(search_page_url):
        driver.get(search_page_url)
        time.sleep(2)

    # -------------------------------
    # Clear previous search
    # -------------------------------
    try:
        clear_buttons = driver.find_elements(By.CSS_SELECTOR, "button.clear-row-button")
        if clear_buttons:
            for btn in clear_buttons:
                try:
                    btn.click()
                    time.sleep(0.5)
                except:
                    continue
    except Exception as e:
        print("Could not clear previous search terms:", e)

    # -------------------------------
    # Fill First and Last Name fields
    # -------------------------------
    try:
        first_name_field = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "input[name='firstName']"))
        )
        first_name_field.clear()
        first_name_field.send_keys(first_name)

        last_name_field = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "input[name='lastName']"))
        )
        last_name_field.clear()
        last_name_field.send_keys(last_name)
    except Exception as e:
        print("Could not fill name fields:", e)
        continue

    # -------------------------------
    # Click Search
    # -------------------------------
    try:
        search_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, 'button[data-ta="run-search"]'))
        )
        search_button.click()
        time.sleep(3)
    except Exception as e:
        print("Could not click Search button:", e)
        continue

    # -------------------------------
    # Select author page if needed
    # -------------------------------
    try:
        current_url = driver.current_url
        if not current_url.startswith("https://www-webofscience-com.emedien.ub.uni-muenchen.de/wos/author/record/"):
            records_list = driver.find_elements(By.CSS_SELECTOR, "app-records-list app-author-summary-record")
            if records_list:
                first_record_link = records_list[0].find_element(By.CSS_SELECTOR, "a.title-link")
                first_record_link.click()
                time.sleep(3)
    except Exception as e:
        print("Error selecting author from results:", e)

    # -------------------------------
    # Scrape author info
    # -------------------------------
    try:
        author_div = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "span.more-details"))
        )
        author_data['further_info'] = author_div.text.strip()
    except:
        author_data['further_info'] = None

    try:
        rid_div = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.author-detail-section-content"))
        )
        colon_span = rid_div.find_element(By.CSS_SELECTOR, "span.colonMark")
        if "Web of Science ResearcherID" in colon_span.text:
            rid_span = colon_span.find_element(By.XPATH, "following-sibling::span")
            author_data['RID'] = rid_span.text.strip()
        else:
            author_data['RID'] = None
    except:
        author_data['RID'] = None

    try:
        subjects_container = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.Subjectcategory-chips.display-data-value"))
        )
        subject_spans = subjects_container.find_elements(By.CSS_SELECTOR, "span.chip-span")
        author_data['subjects'] = [span.text.strip() for span in subject_spans if span.text.strip()]
    except:
        author_data['subjects'] = []

    try:
        summary_section = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.summary-section"))
        )
        summary_items = summary_section.find_elements(By.CSS_SELECTOR, "p.summary-item")
        profile_summary = {}
        for item in summary_items:
            count_span = item.find_element(By.CSS_SELECTOR, "span.summary-count")
            label_span = item.find_element(By.CSS_SELECTOR, "span.summary-label")
            profile_summary[label_span.text.strip()] = count_span.text.strip()
        author_data['profile_summary'] = profile_summary
    except:
        author_data['profile_summary'] = {}

    try:
        metrics_container = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.wat-author-metric-main-div"))
        )
        metric_blocks = metrics_container.find_elements(By.CSS_SELECTOR, "div.wat-author-metric-inline-block")
        author_metrics = {}
        for block in metric_blocks:
            try:
                value_elem = block.find_element(By.CSS_SELECTOR, "div.wat-author-metric")
                label_elem = block.find_element(By.CSS_SELECTOR, "div.wat-author-metric-descriptor")
                sub_label_elem = block.find_elements(By.CSS_SELECTOR, "span.wat-author-metric-sub-descriptor")
                value = value_elem.text.strip()
                label = label_elem.text.strip()
                sub_label = sub_label_elem[0].text.strip() if sub_label_elem else None
                full_label = f"{label} ({sub_label})" if sub_label else label
                author_metrics[full_label] = value
            except:
                continue
        author_data['metrics'] = author_metrics
    except:
        author_data['metrics'] = {}

    # -------------------------------
    # Save author data
    # -------------------------------
    all_authors_data.append(author_data)


    # -------------------------------
    # SAVE CHECKPOINT EVERY 500 AUTHORS
    # -------------------------------
    if (idx + 1) % 100 == 0:
        df_checkpoint = pd.DataFrame(all_authors_data)
        checkpoint_file = os.path.join(path_data, "metadata_wos_acknolwedged_whilescraping.csv")
        df_checkpoint.to_csv(checkpoint_file, index=False)
        print(f"✅ Checkpoint saved after {idx+1} authors: {checkpoint_file}")
    # -------------------------------


    # Return to search page
    if not driver.current_url.startswith(search_page_url):
        driver.get(search_page_url)
        time.sleep(2)

    time.sleep(2)  # small pause before next author

# -------------------------------
# Final dataset
# -------------------------------
df_authors = pd.DataFrame(all_authors_data)
print("\nAll authors scraped ✅")
print(df_authors.head())



Processing ALEXANDRE MAS (1/9338)
Estimating time remaining...

Processing DARON ACEMOGLU (2/9338)
Elapsed: 11.7s | Estimated remaining: 109507.8s (~1825.1 min)

Processing DAVID AUTOR (3/9338)
Elapsed: 27.7s | Estimated remaining: 129397.0s (~2156.6 min)

Processing DAVID DEMING (4/9338)
Elapsed: 40.7s | Estimated remaining: 126615.3s (~2110.3 min)

Processing EMILY OSTER (5/9338)
Elapsed: 51.9s | Estimated remaining: 121092.4s (~2018.2 min)

Processing ESTHER DUFLO (6/9338)
Elapsed: 64.2s | Estimated remaining: 119914.1s (~1998.6 min)

Processing FRANCESCO TREBBI (7/9338)
Elapsed: 76.3s | Estimated remaining: 118701.0s (~1978.3 min)

Processing GAUTAM RAO (8/9338)
Elapsed: 91.8s | Estimated remaining: 122352.6s (~2039.2 min)

Processing GEORGY EGOROV (9/9338)
Elapsed: 106.7s | Estimated remaining: 124461.9s (~2074.4 min)

Processing JESSE SHAPIRO (10/9338)
Elapsed: 119.0s | Estimated remaining: 123377.3s (~2056.3 min)

Processing JOHN FRIEDMAN (11/9338)
Elapsed: 134.4s | Estimated r

In [21]:
print(df_authors.head())


#bryan graham - empty? did not move over search results


# save to CSV
output_file = os.path.join(path_data, "metadata_wos_acknolwedged.csv")
df_authors.to_csv(output_file, index=False)

  first_name last_name                                       further_info  \
0  ALEXANDRE       MAS  Universite Grenoble Alpes (UGA)INRAEGRENOBLE, ...   
1      DARON  ACEMOGLU                                                      
2      DAVID     AUTOR        Massachusetts Institute of Technology (MIT)   
3      DAVID    DEMING  Harvard UniversityHarvard Kennedy SchCAMBRIDGE...   
4      EMILY     OSTER  Brown UniversityBrown University Department of...   

             RID                                           subjects  \
0  DFR-0391-2022  [Business & Economics, Engineering, Water Reso...   
1  JQW-5242-2023  [Business & Economics, Engineering, Mathematic...   
2  OGR-1920-2025  [Business & Economics, Science & Technology - ...   
3  CNZ-5232-2022  [Business & Economics, Geology, Water Resource...   
4  FQY-2513-2022  [Business & Economics, Health Care Sciences & ...   

                                     profile_summary  \
0  {'Total documents': '41', 'Web of Science Core...  